In [ ]:
import os
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, classification_report

# ========================================
# 数据加载与预处理（保留原有部分）
# ========================================
# Define paths
DATA_DIR_FAKE = "C:/Users/M2-Winterfell/Downloads/pulse2pulse_150k/from_006_chkp_2500_150k"
DATA_DIR_REAL = "C:/Users/M2-Winterfell/Downloads/GAN-models-for-Bio-Authentication-through-ECG-signals/datasets/real_ecgs"

# Load data functions (保持原有实现)
def load_lead_I_from_asc(file_path):
    try:
        # Load the .asc file (assuming it's space or tab-delimited)
        ecg_data = np.loadtxt(file_path)

        # Extract Lead I (assuming first column is Lead I)
        lead_I = ecg_data[:, 0]  # Lead I

        return lead_I
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def load_exactly_2000_lead_I_from_directory(data_dir, n=2000):
    all_lead_I = []
    i = 0

    # Initialize the progress bar
    with tqdm(total=n, desc=f"Loading Lead I from {data_dir.split('/')[-1]}", unit='file') as pbar:
        # Load ECG files until we have 2000 valid files
        while len(all_lead_I) < n:
            file_name = f"{i}.asc"
            file_path = os.path.join(data_dir, file_name)

            if os.path.exists(file_path):
                lead_I = load_lead_I_from_asc(file_path)
                if lead_I is not None:
                    all_lead_I.append(lead_I)
                    pbar.update(1)  # Update progress bar when a valid file is loaded

            i += 1

    # Convert list to NumPy array
    all_lead_I = np.array(all_lead_I)

    return all_lead_I

# 加载数据
lead_I_real = load_exactly_2000_lead_I_from_directory(DATA_DIR_REAL, n=2000)
lead_I_fake = load_exactly_2000_lead_I_from_directory(DATA_DIR_FAKE, n=2000)

# 归一化处理
def min_max_normalize(data):
    min_val = np.min(data)
    max_val = np.max(data)
    return 2 * (data - min_val) / (max_val - min_val) - 1

lead_I_real = np.array([min_max_normalize(ecg) for ecg in lead_I_real])
lead_I_fake = np.array([min_max_normalize(ecg) for ecg in lead_I_fake])

# ========================================
# 数据准备（新修改部分）
# ========================================
# 分割真实数据：训练集（70%）、验证集（15%）、测试集（15%）
X_train_real, X_temp_real = train_test_split(lead_I_real, test_size=0.3, random_state=42)
X_val_real, X_test_real = train_test_split(X_temp_real, test_size=0.5, random_state=42)

# 创建测试集（包含真实和假数据）
X_test = np.concatenate([X_test_real, lead_I_fake], axis=0)
y_test = np.concatenate([np.ones(len(X_test_real)), np.zeros(len(lead_I_fake))], axis=0)

# 调整数据形状为LSTM需要的格式 (samples, timesteps, features)
timesteps = X_train_real.shape[1]
features = 1

X_train = X_train_real.reshape(-1, timesteps, features)
X_val = X_val_real.reshape(-1, timesteps, features)
X_test = X_test.reshape(-1, timesteps, features)

# ========================================
# LSTM-Autoencoder模型构建
# ========================================
def build_lstm_autoencoder(input_shape):
    model = Sequential([
        # Encoder
        LSTM(128, activation='tanh', input_shape=input_shape, return_sequences=True),
        Dropout(0.2),
        LSTM(64, activation='tanh', return_sequences=False),
        
        # Bottleneck
        RepeatVector(input_shape[0]),  # 重建时间序列长度
        
        # Decoder
        LSTM(64, activation='tanh', return_sequences=True),
        Dropout(0.2),
        LSTM(128, activation='tanh', return_sequences=True),
        TimeDistributed(Dense(features))
    ])
    return model

# 初始化模型
model = build_lstm_autoencoder((timesteps, features))
model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
model.summary()

# ========================================
# 模型训练
# ========================================
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    X_train, X_train,  # Autoencoder训练目标为重建输入
    epochs=10,
    batch_size=32,
    validation_data=(X_val, X_val),
    callbacks=[early_stop]
)

# ========================================
# 异常检测与评估
# ========================================
def calculate_reconstruction_error(data):
    reconstructions = model.predict(data, verbose=0)
    return np.mean(np.square(data - reconstructions), axis=(1, 2))

# 计算各数据集的重构误差
train_error = calculate_reconstruction_error(X_train)
val_error = calculate_reconstruction_error(X_val)
test_error = calculate_reconstruction_error(X_test)

# 动态阈值设置（使用训练集的统计特性）
threshold = np.mean(train_error) + 3 * np.std(train_error)
print(f"\nAnomaly Detection Threshold: {threshold:.4f}")

# 预测标签（0=异常/假，1=正常/真）
y_pred = (test_error <= threshold).astype(int)

# 评估指标
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Fake", "Real"]))
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# ========================================
# 可视化部分
# ========================================
# 训练过程可视化
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training Progress')
plt.ylabel('MSE Loss')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()
plt.show()

# 重构误差分布可视化
plt.figure(figsize=(10, 6))
plt.hist(test_error[y_test == 0], bins=50, alpha=0.5, label='Fake ECG')
plt.hist(test_error[y_test == 1], bins=50, alpha=0.5, label='Real ECG')
plt.axvline(threshold, color='r', linestyle='--', label=f'Threshold ({threshold:.2f})')
plt.title('Reconstruction Error Distribution')
plt.xlabel('Mean Squared Error')
plt.ylabel('Count (log scale)')
plt.yscale('log')
plt.legend()
plt.show()

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix')
plt.show()

# 信号重建对比可视化
def visualize_reconstruction(samples, labels):
    plt.figure(figsize=(15, 8))
    for i, (sample, label) in enumerate(zip(samples, labels)):
        original = X_test[sample].squeeze()
        reconstructed = model.predict(X_test[sample].reshape(1, timesteps, 1)).squeeze()
        
        plt.subplot(2, 2, i+1)
        plt.plot(original, label='Original', alpha=0.7, linewidth=1.5)
        plt.plot(reconstructed, label='Reconstructed', alpha=0.7, linestyle='--')
        plt.fill_between(np.arange(len(original)), 
                         original, reconstructed,
                         where=(np.abs(original - reconstructed) > threshold**0.5),
                         color='red', alpha=0.3, label='High Error Regions')
        plt.title(f"{'Real' if label == 1 else 'Fake'} ECG (MSE: {test_error[sample]:.4f})")
        plt.xlabel('Time Steps')
        plt.ylabel('Normalized Amplitude')
        plt.legend()
    plt.tight_layout()
    plt.show()

# 随机选择样本可视化
np.random.seed(42)
real_samples = np.random.choice(np.where(y_test == 1)[0], 2)
fake_samples = np.random.choice(np.where(y_test == 0)[0], 2)
visualize_reconstruction(real_samples, [1, 1])
visualize_reconstruction(fake_samples, [0, 0])

# 误差热力图可视化（展示异常区域）
def plot_error_heatmap(sample_index):
    original = X_test[sample_index]
    reconstructed = model.predict(original.reshape(1, timesteps, 1))
    error = np.square(original - reconstructed).squeeze()
    
    plt.figure(figsize=(15, 5))
    plt.subplot(2, 1, 1)
    plt.plot(original.squeeze(), label='ECG Signal')
    plt.title(f"ECG Signal (True: {'Real' if y_test[sample_index] == 1 else 'Fake'})")
    plt.xlabel('Time Steps')
    plt.ylabel('Amplitude')
    
    plt.subplot(2, 1, 2)
    plt.imshow(error.reshape(1, -1), aspect='auto', cmap='hot')
    plt.colorbar(label='Squared Error')
    plt.title('Reconstruction Error Heatmap')
    plt.xlabel('Time Steps')
    plt.yticks([])
    plt.tight_layout()
    plt.show()

# 示例可视化
plot_error_heatmap(real_samples[0])
plot_error_heatmap(fake_samples[0])